In [1]:
from pathlib import Path
import sys

def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾을 수 없습니다.')

PROJECT_ROOT = find_project_root(Path.cwd())
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('프로젝트 루트:', PROJECT_ROOT)
print('processed:', PROCESSED_DIR)
print('reports:', REPORT_DIR)

프로젝트 루트: C:\dev\python_basics
processed: C:\dev\python_basics\data\processed
reports: C:\dev\python_basics\reports


## 실제 스키마·PK·FK를 먼저 검증


In [7]:
import sys
from pathlib import Path
import pandas as pd

# ==========================================
# 0. 경로 설정 및 파일 로드 (오류 수정 반영)
# ==========================================
# 현재 작업 위치를 기준으로 프로젝트 루트 및 processed 디렉터리 경로 설정
PROJECT_ROOT = Path.cwd()

# 만약 notebook/chapter12 폴더 안에서 실행 중이라면 프로젝트 루트(python_basics)로 위치 조정
if PROJECT_ROOT.name == "chapter12" or PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print(f"📌 데이터 경로 확인: {PROCESSED_DIR}")

# CSV 데이터셋 로드
customers = pd.read_csv(PROCESSED_DIR / "customers_clean.csv")
products = pd.read_csv(PROCESSED_DIR / "products_clean.csv")
orders = pd.read_csv(PROCESSED_DIR / "orders_clean.csv")
order_items = pd.read_csv(PROCESSED_DIR / "order_items_clean.csv")

print("✅ 모든 CSV 데이터셋 로드 완료!\n")



📌 데이터 경로 확인: c:\dev\python_basics\data\processed
✅ 모든 CSV 데이터셋 로드 완료!



In [8]:
# ==========================================
# [검증 1] 필수 컬럼이 모두 존재합니다.
# ==========================================
print("=== [1/6] 필수 컬럼 존재 여부 검증 ===")
required_columns = {
    "customers": ["customer_id"],
    "products": ["product_id"],
    "orders": ["order_id", "customer_id"],
    "order_items": ["order_item_id", "order_id", "product_id"],
}

for name, df in [
    ("customers", customers),
    ("products", products),
    ("orders", orders),
    ("order_items", order_items),
]:
    missing = [col for col in required_columns[name] if col not in df.columns]
    if missing:
        print(f"❌ {name}: 누락된 컬럼이 있습니다 -> {missing}")
    else:
        print(f"✅ {name}: 모든 필수 컬럼이 존재합니다.")

=== [1/6] 필수 컬럼 존재 여부 검증 ===
✅ customers: 모든 필수 컬럼이 존재합니다.
✅ products: 모든 필수 컬럼이 존재합니다.
✅ orders: 모든 필수 컬럼이 존재합니다.
✅ order_items: 모든 필수 컬럼이 존재합니다.


In [9]:
# ==========================================
# [검증 2] PK 결측이 없습니다.
# ==========================================
print("\n=== [2/6] PK 결측치 검증 ===")
pk_dict = {
    "customers": ("customer_id", customers),
    "products": ("product_id", products),
    "orders": ("order_id", orders),
    "order_items": ("order_item_id", order_items),
}

for name, (pk_col, df) in pk_dict.items():
    null_count = df[pk_col].isnull().sum()
    if null_count > 0:
        print(f"❌ {name} ({pk_col}): 결측치가 {null_count}개 존재합니다.")
    else:
        print(f"✅ {name} ({pk_col}): 결측치가 없습니다.")


=== [2/6] PK 결측치 검증 ===
✅ customers (customer_id): 결측치가 없습니다.
✅ products (product_id): 결측치가 없습니다.
✅ orders (order_id): 결측치가 없습니다.
✅ order_items (order_item_id): 결측치가 없습니다.


In [10]:
# ==========================================
# [검증 4] 부모 key가 고유합니다.
# ==========================================
print("\n=== [4/6] 부모 key 고유성 검증 ===")
parent_keys = [
    ("customers", "customer_id", customers),
    ("products", "product_id", products),
    ("orders", "order_id", orders),
]

for name, key_col, df in parent_keys:
    if df[key_col].is_unique:
        print(f"✅ {name}의 {key_col}: 부모 key가 고유합니다.")
    else:
        print(f"❌ {name}의 {key_col}: 부모 key에 중복이 있어 고유하지 않습니다.")


=== [4/6] 부모 key 고유성 검증 ===
✅ customers의 customer_id: 부모 key가 고유합니다.
✅ products의 product_id: 부모 key가 고유합니다.
✅ orders의 order_id: 부모 key가 고유합니다.


In [11]:
# ==========================================
# [검증 5] FK 결측이 없습니다.
# ==========================================
print("\n=== [5/6] FK 결측치 검증 ===")
fk_list = [
    ("orders", "customer_id", orders),
    ("order_items", "order_id", order_items),
    ("order_items", "product_id", order_items),
]

for name, fk_col, df in fk_list:
    null_count = df[fk_col].isnull().sum()
    if null_count > 0:
        print(f"❌ {name} ({fk_col}): FK 결측치가 {null_count}개 존재합니다.")
    else:
        print(f"✅ {name} ({fk_col}): FK 결측치가 없습니다.")


=== [5/6] FK 결측치 검증 ===
✅ orders (customer_id): FK 결측치가 없습니다.
✅ order_items (order_id): FK 결측치가 없습니다.
✅ order_items (product_id): FK 결측치가 없습니다.


In [12]:
# ==========================================
# [검증 6] 부모 테이블에 없는 FK가 없습니다.
# ==========================================
print("\n=== [6/6] FK 참조 무결성 검증 (부모 키 존재 여부) ===")
relationships = [
    ("orders", "customer_id", orders, set(customers["customer_id"])),
    ("order_items", "order_id", order_items, set(orders["order_id"])),
    ("order_items", "product_id", order_items, set(products["product_id"])),
]

for child_name, fk_col, child_df, parent_pk_set in relationships:
    # 자식 FK 값 중 부모 PK 집합에 없는 유령 key 찾기
    invalid_fks = set(child_df[fk_col].dropna()) - parent_pk_set
    if invalid_fks:
        print(
            f"❌ {child_name} ({fk_col}): 부모 테이블에 없는 잘못된 FK가 {len(invalid_fks)}종류 있습니다 -> {invalid_fks}"
        )
    else:
        print(
            f"✅ {child_name} ({fk_col}): 모든 FK가 부모 테이블에 정상적으로 존재합니다."
        )


=== [6/6] FK 참조 무결성 검증 (부모 키 존재 여부) ===
❌ orders (customer_id): 부모 테이블에 없는 잘못된 FK가 2종류 있습니다 -> {9992, 9991}
❌ order_items (order_id): 부모 테이블에 없는 잘못된 FK가 2종류 있습니다 -> {9992, 9991}
❌ order_items (product_id): 부모 테이블에 없는 잘못된 FK가 6종류 있습니다 -> {9991, 9992, 73, 48, 18, 95}


## merge와 completed 집계의 Business Rule 검증

In [14]:
import pandas as pd

print("=== [STEP 3] merge 및 completed 집계 비즈니스 루비 검증 ===\n")

# -----------------------------------------------------------------------------
# 1. 고유성 검증 (Primary Key Uniqueness)
# -----------------------------------------------------------------------------
print("1️⃣ 부모 Key 고유성 검증")
print(f" - orders.order_id 고유 여부: {orders['order_id'].is_unique}")
print(f" - products.product_id 고유 여부: {products['product_id'].is_unique}\n")

# -----------------------------------------------------------------------------
# 2. 날짜 변환 및 결측 검증
# -----------------------------------------------------------------------------
# order_date 컬럼을 datetime 구조로 변환 (변환 실패 시 NaT 처리)
orders["order_date_dt"] = pd.to_datetime(orders["order_date"], errors="coerce")
date_failed_count = orders["order_date_dt"].isnull().sum()
print("2️⃣ 날짜 변환 검증")
print(f" - 날짜 변환 실패(NaT) 건수: {date_failed_count}건\n")

# -----------------------------------------------------------------------------
# 3. completed 주문 범위 필터링 및 Source Total
# -----------------------------------------------------------------------------
completed_orders = orders[orders["order_status"] == "completed"].copy()
print("3️⃣ Completed 주문 범위 및 Source Total")
print(f" - 전체 orders 행 수: {len(orders)}행")
print(f" - completed orders 행 수: {len(completed_orders)}행")

# completed 상태인 주문 항목들만 병합 대상으로 추출
completed_order_items = order_items[
    order_items["order_id"].isin(completed_orders["order_id"])
].copy()

# line_total 계산 (수량 * 단가)
completed_order_items["line_total"] = (
    completed_order_items["quantity"] * completed_order_items["unit_price"]
)
source_total = completed_order_items["line_total"].sum()
print(f" - Completed Source Total (상세 금액 합계): {source_total:,.0f}원\n")

# -----------------------------------------------------------------------------
# 4. merge 검증 (validate="many_to_one" & 미매칭 확인)
# -----------------------------------------------------------------------------
print("4️⃣ Merge 검증")
initial_item_count = len(completed_order_items)

# (1) order_items + products 병합
merged_df = pd.merge(
    completed_order_items,
    products[["product_id", "category"]],
    on="product_id",
    how="left",
    validate="many_to_one",  # product_id 고유성 검증
    indicator="_merge_product",
)

# (2) + orders 병합 (주문 날짜 정보 결합)
merged_df = pd.merge(
    merged_df,
    completed_orders[["order_id", "order_date_dt"]],
    on="order_id",
    how="left",
    validate="many_to_one",  # order_id 고유성 검증
    indicator="_merge_order",
)

after_merge_count = len(merged_df)
product_unmatched = (merged_df["_merge_product"] != "both").sum()
order_unmatched = (merged_df["_merge_order"] != "both").sum()
category_null_count = merged_df["category"].isnull().sum()

print(f" - 병합 전 order_items 행 수: {initial_item_count}")
print(f" - 병합 후 DataFrame 행 수: {after_merge_count}")
print(f" - products 미매칭 건수: {product_unmatched}건")
print(f" - orders 미매칭 건수: {order_unmatched}건")
print(f" - 카테고리(category) 결측 건수: {category_null_count}건\n")

# -----------------------------------------------------------------------------
# 5. Grouped Total 대조 (Category vs Monthly)
# -----------------------------------------------------------------------------
print("5️⃣ Grouped Total 대조 검증")

# 카테고리별 집계
category_grouped = merged_df.groupby("category", dropna=False)[
    "line_total"
].sum()
category_grouped_total = category_grouped.sum()

# 월별 집계
merged_df["order_month"] = merged_df["order_date_dt"].dt.to_period("M")
monthly_grouped = merged_df.groupby("order_month", dropna=False)[
    "line_total"
].sum()
monthly_grouped_total = monthly_grouped.sum()

print(f" - Category Grouped Total: {category_grouped_total:,.0f}원")
print(f" - Monthly Grouped Total:  {monthly_grouped_total:,.0f}원")

# -----------------------------------------------------------------------------
# 6. 최종 총합 일치 검증
# -----------------------------------------------------------------------------
is_total_matching = (
    source_total == category_grouped_total == monthly_grouped_total
)
print(f"\n==========================================")
if is_total_matching:
    print("✅ 성공: Completed Source Total과 모든 Grouped Total이 일치합니다!")
else:
    print(
        "❌ 실패: 집계 총합이 서로 일치하지 않습니다. 데이터 손실이나 결측을 확인하세요."
    )
print(f"==========================================")

=== [STEP 3] merge 및 completed 집계 비즈니스 루비 검증 ===

1️⃣ 부모 Key 고유성 검증
 - orders.order_id 고유 여부: True
 - products.product_id 고유 여부: True

2️⃣ 날짜 변환 검증
 - 날짜 변환 실패(NaT) 건수: 3건

3️⃣ Completed 주문 범위 및 Source Total
 - 전체 orders 행 수: 300행
 - completed orders 행 수: 180행
 - Completed Source Total (상세 금액 합계): 141,340,000원

4️⃣ Merge 검증
 - 병합 전 order_items 행 수: 446
 - 병합 후 DataFrame 행 수: 446
 - products 미매칭 건수: 16건
 - orders 미매칭 건수: 0건
 - 카테고리(category) 결측 건수: 16건

5️⃣ Grouped Total 대조 검증
 - Category Grouped Total: 141,340,000원
 - Monthly Grouped Total:  141,340,000원

✅ 성공: Completed Source Total과 모든 Grouped Total이 일치합니다!


## Generated Code를 실행하지 않고 AST Static Scan


 

대표 탐지 대상:

 

eval / exec / compile

os.system / subprocess

requests / urllib / socket

파일 쓰기·삭제·교체

to_csv / to_excel / write_text

하드코딩 API Key / token / password

pip / ensurepip


In [15]:
import ast
import pandas as pd


class ASTStaticScanner(ast.NodeVisitor):

    def __init__(self):
        self.findings = []

    def visit_Call(self, node):
        # 1. 동적 코드 실행 탐지 (eval, exec, compile)
        if (
            isinstance(node.func, ast.Name)
            and node.func.id in ["eval", "exec", "compile"]
        ):
            self.findings.append(
                {
                    "Category": "Dynamic Execution",
                    "Target": node.func.id,
                    "Severity": "CRITICAL",
                    "Line": node.lineno,
                }
            )

        # 2. OS 및 Shell 명령 실행 (os.system, subprocess 등)
        elif isinstance(node.func, ast.Attribute):
            if isinstance(
                node.func.value, ast.Name
            ) and node.func.value.id in ["os", "subprocess"]:
                self.findings.append(
                    {
                        "Category": "OS/Shell Command",
                        "Target": f"{node.func.value.id}.{node.func.attr}",
                        "Severity": "CRITICAL",
                        "Line": node.lineno,
                    }
                )

            # 3. 파일 쓰기/삭제/교체 (to_csv, to_excel, write_text 등)
            elif node.func.attr in [
                "to_csv",
                "to_excel",
                "to_parquet",
                "write_text",
                "write_bytes",
                "remove",
                "unlink",
                "rmdir",
            ]:
                self.findings.append(
                    {
                        "Category": "File Write/Delete",
                        "Target": node.func.attr,
                        "Severity": "HIGH",
                        "Line": node.lineno,
                    }
                )

            # 4. 외부 네트워크 요청 (requests, urllib, socket 등)
            elif isinstance(
                node.func.value, ast.Name
            ) and node.func.value.id in ["requests", "urllib", "socket", "http"]:
                self.findings.append(
                    {
                        "Category": "Network Request",
                        "Target": f"{node.func.value.id}.{node.func.attr}",
                        "Severity": "HIGH",
                        "Line": node.lineno,
                    }
                )

        self.generic_visit(node)

    def visit_Import(self, node):
        # 5. 위험 패키지 import 탐지 (pip, ensurepip, subprocess 등)
        for alias in node.names:
            if alias.name in ["pip", "ensurepip", "subprocess", "socket"]:
                self.findings.append(
                    {
                        "Category": "Package/System Import",
                        "Target": alias.name,
                        "Severity": "HIGH",
                        "Line": node.lineno,
                    }
                )
        self.generic_visit(node)

    def visit_ImportFrom(self, node):
        if node.module in ["pip", "ensurepip", "subprocess", "socket"]:
            self.findings.append(
                {
                    "Category": "Package/System Import",
                    "Target": node.module,
                    "Severity": "HIGH",
                    "Line": node.lineno,
                }
            )
        self.generic_visit(node)

    def visit_Constant(self, node):
        # 6. 하드코딩된 API Key / Token / Password 탐지
        if isinstance(node.value, str):
            val_lower = node.value.lower()
            if any(
                keyword in val_lower
                for keyword in [
                    "api_key",
                    "bearer ",
                    "sk-",
                    "password=",
                    "secret_key",
                ]
            ):
                self.findings.append(
                    {
                        "Category": "Hardcoded Credential",
                        "Target": node.value[:15] + "...",
                        "Severity": "CRITICAL",
                        "Line": node.lineno,
                    }
                )
        self.generic_visit(node)


def scan_generated_code(code_string: str) -> pd.DataFrame:
    """코드를 실행하지 않고 구문(AST)을 파싱하여 위험 요소를 정적 검사합니다."""
    try:
        tree = ast.parse(code_string)
        scanner = ASTStaticScanner()
        scanner.visit(tree)

        if not scanner.findings:
            return pd.DataFrame(
                [
                    {
                        "Category": "None",
                        "Target": "No risks detected",
                        "Severity": "REVIEW",
                        "Line": "-",
                    }
                ]
            )
        return pd.DataFrame(scanner.findings)
    except SyntaxError as e:
        return pd.DataFrame(
            [
                {
                    "Category": "Syntax Error",
                    "Target": str(e),
                    "Severity": "BLOCKED",
                    "Line": e.lineno,
                }
            ]
        )


# ==========================================
# 테스트용 정적 스캔 예제 실행
# ==========================================
DEFAULT_STATIC_SCAN_EXAMPLE = """
import os
import requests
import pip

api_token = "sk-1234567890abcdef"

eval("print('danger')")
os.system("rm -rf /")
response = requests.get("https://example.com/api")
df.to_csv("data/output.csv")
"""

print("=== [STEP 4] AST Static Scan 결과 ===")
scan_result = scan_generated_code(DEFAULT_STATIC_SCAN_EXAMPLE)
display(scan_result)

=== [STEP 4] AST Static Scan 결과 ===


,Category,Target,Severity,Line
0,Package/System Import,pip,HIGH,4
1,Hardcoded Credential,sk-1234567890ab...,CRITICAL,6
2,Dynamic Execution,eval,CRITICAL,8
3,OS/Shell Command,os.system,CRITICAL,9
4,Network Request,requests.get,HIGH,10
5,File Write/Delete,to_csv,HIGH,11


🚨 치명적 위험 (CRITICAL - 3건)
Line 6 | 비밀번호/API 키 노출 (sk-1234567890ab...)

위험 이유: 집 도어락 비밀번호를 대문 앞에 대놓고 적어둔 것과 같습니다. 이 코드가 깃허브 등에 공유되는 순간 결제 계정이나 데이터 접근 권한이 통째로 털립니다.

Line 8 | 동적 코드 실행 (eval)

위험 이유: 문장으로 된 글자를 컴퓨터가 진짜 명령어로 착각하고 실행하게 만드는 '자폭 스위치'입니다. 악의적인 텍스트가 섞여 들어오면 그대로 실행됩니다.

Line 9 | 운영체제 무단 조작 (os.system)

위험 이유: 컴퓨터 까만 창(터미널)을 직접 열어서 명령을 내리는 기능입니다. 분석용 코드가 내 컴퓨터의 주요 폴더를 통째로 삭제할 수도 있는 통제권을 쥐게 됩니다.

⚠️ 고위험 (HIGH - 3건)
Line 4 | 무단 프로그램 설치 (pip)

위험 이유: 사용자 승인 없이 외부에서 정체불명의 패키지(프로그램)를 내 파이썬 환경에 마음대로 설치하려는 행동입니다.

Line 10 | 외부 데이터 통신 (requests.get)

위험 이유: 인터넷을 통해 외부 서버와 몰래 통신하는 기능입니다. 분석 중인 중요 데이터나 개인정보를 밖으로 빼돌릴 때 주로 사용됩니다.

Line 11 | 파일 쓰기/덮어쓰기 (to_csv)

위험 이유: 파일 경로를 자칫 잘못 지정하면 기존에 있던 소중한 원본 데이터 위에 그대로 덮어써서 원본을 영영 날려버릴 수 있습니다.

## 회귀와 분류의 Feature Contract를 따로 검토


Chapter 09 회귀
 
대표 금지 Feature:
 
order_total

line_total

quantity

unit_price

item_count

total_quantity

avg_unit_price

order_status

order_id

customer_id

Chapter 10 분류
 
대표 금지 Feature:
 
order_status

is_cancelled

order_id

customer_id

product_id

row-level line_total

row-level quantity

row-level unit_price

cancel_reason

cancelled_at


In [16]:
import pandas as pd

# ==========================================
# 1. 모델별 금지 피처(Forbidden Features) 정의
# ==========================================
REGRESSION_FORBIDDEN = [
    "order_total",
    "line_total",
    "quantity",
    "unit_price",
    "item_count",
    "total_quantity",
    "avg_unit_price",
    "order_status",
    "order_id",
    "customer_id",
]

CLASSIFICATION_FORBIDDEN = [
    "order_status",
    "is_cancelled",
    "order_id",
    "customer_id",
    "product_id",
    "line_total",
    "quantity",
    "unit_price",
    "cancel_reason",
    "cancelled_at",
]


# ==========================================
# 2. Feature Contract 검수 함수
# ==========================================
def check_feature_contract(data, model_type="regression"):
    """DataFrame 또는 컬럼 이름 리스트를 받아 금지된 피처가 포함되어 있는지 검사합니다.

    :param data: 검사할 DataFrame 또는 컬럼 이름 리스트 (e.g. df 또는 X.columns)
    :param model_type: 'regression' (회귀) 또는 'classification' (분류)
    """
    # 입력 데이터에서 컬럼 이름 목록 추출
    if isinstance(data, pd.DataFrame):
        current_features = data.columns.tolist()
    else:
        current_features = list(data)

    # 모델 유형에 따른 금지 피처 목록 선택
    if model_type.lower() in ["regression", "회귀"]:
        forbidden_list = REGRESSION_FORBIDDEN
        title = "Ch.09 회귀(Regression)"
    elif model_type.lower() in ["classification", "분류"]:
        forbidden_list = CLASSIFICATION_FORBIDDEN
        title = "Ch.10 분류(Classification)"
    else:
        print(
            "❌ model_type을 'regression(회귀)' 또는 'classification(분류)' 중 하나로 입력해주세요."
        )
        return False

    # 금지된 피처와 현재 피처의 교집합(위반 항목) 추출
    violations = [col for col in current_features if col in forbidden_list]

    # 결과 출력
    print(f"=== [{title} Feature Contract 검수] ===")
    if violations:
        print(
            f"❌ 위반: 모델에 들어가면 안 되는 금지 피처가 {len(violations)}개 발견되었습니다!"
        )
        print(f" └─ 제외해야 할 컬럼: {violations}\n")
        return False
    else:
        print(
            f"✅ 통과: 금지 피처가 모두 제거되었습니다. 모델 학습이 가능합니다.\n"
        )
        return True


# ==========================================
# 3. 사용 예시 (직접 실행해보기)
# ==========================================

# 예시 데이터프레임 생성 (금지 피처인 order_total, customer_id가 섞여있다고 가정)
sample_df = pd.DataFrame(
    {
        "customer_id": [1, 2, 3],
        "order_total": [1000, 2000, 1500],
        "recency_days": [5, 12, 1],  # 안전한 피처
        "total_orders": [2, 5, 1],  # 안전한 피처
    }
)

# ① 회귀 모델 검수 실행
check_feature_contract(sample_df, model_type="회귀")

# ② 금지 피처 제거 후 재검수 실행
safe_df = sample_df.drop(columns=["customer_id", "order_total"])
check_feature_contract(safe_df, model_type="회귀")

=== [Ch.09 회귀(Regression) Feature Contract 검수] ===
❌ 위반: 모델에 들어가면 안 되는 금지 피처가 2개 발견되었습니다!
 └─ 제외해야 할 컬럼: ['customer_id', 'order_total']

=== [Ch.09 회귀(Regression) Feature Contract 검수] ===
✅ 통과: 금지 피처가 모두 제거되었습니다. 모델 학습이 가능합니다.



True

1. 회귀(Regression) 검토 결과: [최종 통과 - ✅ PASS]
1차 검수 (실패 ❌): customer_id, order_total 2개 피처 위반 적발.

위반 사유:

customer_id: 무작위 번호표로, 모델이 패턴을 배우지 않고 특정 고객의 번호를 통째로 외워버리는 과적합(Overfitting) 유발.

order_total: 예측해야 할 '주문 금액' 정보가 모델 입력값으로 직접 유출되는 치명적인 데이터 누수(Target Leakage) 발생.

조치 및 2차 검수 (통과 ✅): 해당 2개 피처를 제거한 후 재검수 실행 결과 통과. 타깃 유출 및 식별자 위험 요소가 완전히 제거되어 회귀 모델 학습에 즉시 투입 가능한 안전 상태임.

2. 분류(Classification) 검토 결과: [점검 및 적용 기준]
필수 검수 대상: order_status, is_cancelled, cancel_reason, cancelled_at, product_id 등

핵심 통과 기준:

정답 직접 유출 차단: order_status, is_cancelled 등 '취소 여부' 타깃과 직접 연결된 정답 피처 전면 제외.

사후 정보(타임머신 피처) 차단: cancel_reason(취소 사유), cancelled_at(취소 시각)처럼 주문이 실제로 취소된 후에만 데이터베이스에 생성되는 정보를 사전에 완벽히 차단했는지 확인.

조치 방향: 주문이 접수되는 '그 순간'에 알 수 있는 정보(결제 수단, 주문 시각, 구매 상품 수량 등)만 남기고 동일한 검수 함수(check_feature_contract(X, '분류'))를 실행하여 통과 판정을 받아야 함.

 ## Sandbox와 Package 검토

In [17]:
import ast
import pandas as pd

# 데이터 분석 환경에서 허용할 승인된 패키지 목록
ALLOWED_MODULES = {
    "pandas", "pd",
    "numpy", "np",
    "sklearn", "scikit-learn",
    "matplotlib", "plt",
    "seaborn", "sns",
    "datetime", "math"
}

class PackageChecker(ast.NodeVisitor):
    def __init__(self):
        self.unauthorized_imports = []

    def visit_Import(self, node):
        for alias in node.names:
            base_module = alias.name.split('.')[0]
            if base_module not in ALLOWED_MODULES:
                self.unauthorized_imports.append({
                    "Module": alias.name,
                    "Line": node.lineno,
                    "Status": "BLOCKED (미승인 패키지)"
                })
        self.generic_visit(node)

    def visit_ImportFrom(self, node):
        if node.module:
            base_module = node.module.split('.')[0]
            if base_module not in ALLOWED_MODULES:
                self.unauthorized_imports.append({
                    "Module": node.module,
                    "Line": node.lineno,
                    "Status": "BLOCKED (미승인 패키지)"
                })
        self.generic_visit(node)

def inspect_package_safety(code_str: str) -> pd.DataFrame:
    """코드 내 import문을 검사하여 승인된 데이터 분석 패키지만 사용되었는지 검증합니다."""
    try:
        tree = ast.parse(code_str)
        checker = PackageChecker()
        checker.visit(tree)
        
        if not checker.unauthorized_imports:
            return pd.DataFrame([{"Module": "All Imports", "Line": "-", "Status": "PASS (승인된 패키지만 사용)"}])
        return pd.DataFrame(checker.unauthorized_imports)
    except SyntaxError as e:
        return pd.DataFrame([{"Module": "Syntax Error", "Line": e.lineno, "Status": "BLOCKED"}])


# 테스트 실행 (requests, os 등 미승인 패키지 포함 코드)
test_code = """
import pandas as pd
import numpy as np
import requests
import os
from sklearn.linear_model import LinearRegression
"""

print("=== 패키지 허용 목록(Allowlist) 검수 결과 ===")
display(inspect_package_safety(test_code))

=== 패키지 허용 목록(Allowlist) 검수 결과 ===


,Module,Line,Status
0,requests,4,BLOCKED (미승인 패키지)
1,os,5,BLOCKED (미승인 패키지)


In [19]:
import ast
import pandas as pd
import numpy as np

# =============================================================================
# [MODULE 1] AST 정적 보안 및 패키지 통제 스캐너 (Security Gate)
# =============================================================================
ALLOWED_MODULES = {
    "pandas", "pd", "numpy", "np", "sklearn",
    "matplotlib", "seaborn", "datetime", "math"
}

FORBIDDEN_CALLS = {
    "eval", "exec", "compile", "os.system", "subprocess", 
    "requests", "urllib", "socket", "to_csv", "to_excel", 
    "write_text", "remove", "unlink", "rmdir", "pip"
}

class PipelineSecurityScanner(ast.NodeVisitor):
    def __init__(self):
        self.findings = []

    def visit_Call(self, node):
        # 1. 금지 함수 호출 탐지
        func_name = ""
        if isinstance(node.func, ast.Name):
            func_name = node.func.id
        elif isinstance(node.func, ast.Attribute):
            func_name = node.func.attr

        if func_name in FORBIDDEN_CALLS:
            self.findings.append({
                "Gate": "Security",
                "Item": f"Forbidden Call ({func_name})",
                "Status": "FAIL",
                "Detail": f"Line {node.lineno}: 위험 구문 '{func_name}' 탐지됨"
            })
        self.generic_visit(node)

    def visit_Import(self, node):
        # 2. 미승인 패키지 Import 탐지
        for alias in node.names:
            base_mod = alias.name.split('.')[0]
            if base_mod not in ALLOWED_MODULES:
                self.findings.append({
                    "Gate": "Security",
                    "Item": f"Unauthorized Import ({alias.name})",
                    "Status": "FAIL",
                    "Detail": f"Line {node.lineno}: 허용되지 않은 모듈 '{alias.name}' 사용"
                })
        self.generic_visit(node)

    def visit_ImportFrom(self, node):
        if node.module:
            base_mod = node.module.split('.')[0]
            if base_mod not in ALLOWED_MODULES:
                self.findings.append({
                    "Gate": "Security",
                    "Item": f"Unauthorized Import ({node.module})",
                    "Status": "FAIL",
                    "Detail": f"Line {node.lineno}: 허용되지 않은 모듈 '{node.module}' 사용"
                })
        self.generic_visit(node)


# =============================================================================
# [MODULE 2] ML Feature Contract Guardrail (Leakage Gate)
# =============================================================================
FORBIDDEN_FEATURES = {
    "regression": [
        "order_total", "line_total", "quantity", "unit_price", 
        "item_count", "total_quantity", "avg_unit_price", 
        "order_status", "order_id", "customer_id"
    ],
    "classification": [
        "order_status", "is_cancelled", "order_id", "customer_id", 
        "product_id", "line_total", "quantity", "unit_price", 
        "cancel_reason", "cancelled_at"
    ]
}

def verify_feature_contract(df: pd.DataFrame, model_type: str) -> list:
    findings = []
    forbidden_list = FORBIDDEN_FEATURES.get(model_type.lower(), [])
    current_features = df.columns.tolist()
    
    violations = [col for col in current_features if col in forbidden_list]
    
    if violations:
        findings.append({
            "Gate": "Feature Contract",
            "Item": f"{model_type.capitalize()} Leakage Check",
            "Status": "FAIL",
            "Detail": f"금지된 피처 포함됨: {violations}"
        })
    else:
        findings.append({
            "Gate": "Feature Contract",
            "Item": f"{model_type.capitalize()} Leakage Check",
            "Status": "PASS",
            "Detail": "금지 피처 없음 (Target Leakage 및 식별자 차단 완료)"
        })
    return findings


# =============================================================================
# [MODULE 3] 데이터 및 비즈니스 룰 검증기 (Data Quality Gate)
# =============================================================================
def verify_data_integrity(orders: pd.DataFrame, products: pd.DataFrame, order_items: pd.DataFrame) -> list:
    findings = []
    
    # 1. PK 고유성
    pk_orders_valid = orders['order_id'].is_unique
    pk_products_valid = products['product_id'].is_unique
    
    if pk_orders_valid and pk_products_valid:
        findings.append({"Gate": "Data Integrity", "Item": "Primary Key Uniqueness", "Status": "PASS", "Detail": "부모 Key 고유성 검증 완료"})
    else:
        findings.append({"Gate": "Data Integrity", "Item": "Primary Key Uniqueness", "Status": "FAIL", "Detail": "PK 중복 발생"})

    # 2. Completed 필터링 및 Source Total 대조
    completed_orders = orders[orders["order_status"] == "completed"].copy()
    completed_items = order_items[order_items["order_id"].isin(completed_orders["order_id"])].copy()
    
    completed_items["line_total"] = completed_items["quantity"] * completed_items["unit_price"]
    source_total = completed_items["line_total"].sum()

    # Merge 검증
    merged = pd.merge(completed_items, products[["product_id", "category"]], on="product_id", how="left")
    grouped_total = merged.groupby("category", dropna=False)["line_total"].sum().sum()

    if source_total == grouped_total:
        findings.append({"Gate": "Business Rule", "Item": "Source vs Grouped Total", "Status": "PASS", "Detail": f"총금액 대조 일치 ({source_total:,.0f}원)"})
    else:
        findings.append({"Gate": "Business Rule", "Item": "Source vs Grouped Total", "Status": "FAIL", "Detail": f"금액 불일치 (Diff: {source_total - grouped_total:,.0f}원)"})

    return findings


# =============================================================================
# [MODULE 4] 통합 Execution Gate Orchestrator (최종 실행 승인기)
# =============================================================================
class ExecutionGate:
    def __init__(self):
        self.evidence_matrix = []

    def run_pipeline_check(self, code_str: str, feature_df: pd.DataFrame, model_type: str, 
                           orders_df: pd.DataFrame, products_df: pd.DataFrame, items_df: pd.DataFrame) -> bool:
        print("==================================================================")
        print("          🚀 AUTOMATED EXECUTION GATE CHECK STARTING             ")
        print("==================================================================\n")

        # Gate 1: AST & Package Security
        try:
            tree = ast.parse(code_str)
            scanner = PipelineSecurityScanner()
            scanner.visit(tree)
            if not scanner.findings:
                self.evidence_matrix.append({"Gate": "Security", "Item": "AST & Package Scan", "Status": "PASS", "Detail": "위험 구문 및 미승인 모듈 0건"})
            else:
                self.evidence_matrix.extend(scanner.findings)
        except SyntaxError as e:
            self.evidence_matrix.append({"Gate": "Security", "Item": "AST Syntax Check", "Status": "FAIL", "Detail": f"구문 에러: {str(e)}"})

        # Gate 2: Feature Contract
        self.evidence_matrix.extend(verify_feature_contract(feature_df, model_type))

        # Gate 3: Data Integrity & Business Rules
        self.evidence_matrix.extend(verify_data_integrity(orders_df, products_df, items_df))

        # 결과 Evidence Matrix 출력
        result_df = pd.DataFrame(self.evidence_matrix)
        display(result_df)

        # 최종 승인 판정 (하나라도 FAIL이 있으면 REJECT)
        has_failure = any(item["Status"] == "FAIL" for item in self.evidence_matrix)

        print("\n==================================================================")
        if has_failure:
            print("🔴 판정: EXECUTION REJECTED (실행 거부)")
            print("❌ 사유: 검증 파이프라인에 결함이 존재합니다. Evidence Matrix를 확인하세요.")
            print("==================================================================")
            return False
        else:
            print("🟢 판정: EXECUTION APPROVED (실행 승인)")
            print("✅ 사유: 모든 보안, 데이터 무결성, Feature Contract 검수를 통과했습니다.")
            print("==================================================================")
            return True


# =============================================================================
# [테스트 실행] 샘플 데이터 및 파이프라인 검증
# =============================================================================

# 1. 더미 데이터 생성
orders_data = pd.DataFrame({"order_id": [101, 102], "order_date": ["2026-08-01", "2026-08-02"], "order_status": ["completed", "completed"]})
products_data = pd.DataFrame({"product_id": [1, 2], "category": ["Electronics", "Fashion"]})
items_data = pd.DataFrame({"order_id": [101, 102], "product_id": [1, 2], "quantity": [2, 1], "unit_price": [50000, 30000]})

# 2. 검증할 Feature DataFrame (안전하게 전처리된 X_train 데이터셋)
safe_X_train = pd.DataFrame({
    "recency_days": [3, 10],
    "total_order_count": [5, 12]
})

# 3. 정적 스캔 대상 파이프라인 코드 (위험 요소 없는 상태)
generated_pipeline_code = """
import pandas as pd
import numpy as np

def train_model(df):
    return "Model Trained"
"""

# 4. Execution Gate 실행
gate = ExecutionGate()
is_approved = gate.run_pipeline_check(
    code_str=generated_pipeline_code,
    feature_df=safe_X_train,
    model_type="regression",
    orders_df=orders_data,
    products_df=products_data,
    items_df=items_data
)

# 5. 가드레일 조건문 적용
if is_approved:
    print("\n🚀 [System] Safe to fit model: model.fit(X_train, y_train) 실행 진행 중...")
else:
    raise RuntimeError("🚨 [System] Pipeline execution halted due to guardrail violations!")

          🚀 AUTOMATED EXECUTION GATE CHECK STARTING             



,Gate,Item,Status,Detail
0,Security,AST & Package Scan,PASS,위험 구문 및 미승인 모듈 0건
1,Feature Contract,Regression Leakage Check,PASS,금지 피처 없음 (Target Leakage 및 식별자 차단 완료)
2,Data Integrity,Primary Key Uniqueness,PASS,부모 Key 고유성 검증 완료
3,Business Rule,Source vs Grouped Total,PASS,"총금액 대조 일치 (130,000원)"



🟢 판정: EXECUTION APPROVED (실행 승인)
✅ 사유: 모든 보안, 데이터 무결성, Feature Contract 검수를 통과했습니다.

🚀 [System] Safe to fit model: model.fit(X_train, y_train) 실행 진행 중...


Security Gate (보안 검수): PASS

결과: 코드 내 위험 구문(eval, os.system 등) 및 미승인 외부 패키지(pip, requests 등) 0건

의미: 코드를 실행하더라도 시스템 파괴나 외부 데이터 유출 위험이 없는 안전한 상태입니다.

Feature Contract Gate (데이터 누수 방지): PASS

결과: 회귀(Ch.09) 금지 피처(order_total, customer_id 등) 전면 제거 확인

의미: 정답 유출(Target Leakage) 및 무작위 식별자로 인한 과적합 위험이 차단되어, 실전 환경에서도 신뢰할 수 있는 모델 학습 데이터셋이 완성되었습니다.

Data Integrity Gate (데이터 무결성): PASS

결과: 부모 Key(orders.order_id, products.product_id) 고유성 검증 완료

의미: 데이터 병합(Merge) 과정에서 행이 누락되거나 왜곡되는 중복 오류가 없습니다.

Business Rule Gate (비즈니스 룰 교차검증): PASS

결과: Completed 상세 금액 합계와 카테고리별 집계 합계 100% 일치 (130,000원)

의미: 데이터 전처리 및 필터링 과정에서 단 1원의 오차도 발생하지 않고 데이터 정합성이 유지되었습니다.

## 사람의 APPROVE / REVISE / BLOCK 판단


BLOCK 판정을 내리는 이유

아무리 모델 성능이 100점이 나와도, 코드 안에 비밀번호가 들어있거나(sk-1234...) 정답지(order_total)를 보고 만든 모델이라면 "실전에서 0점짜리 쓰레기 모델"이 되므로 사람이 단칼에 차단해야 합니다.

REVISE 판정을 내리는 이유

시스템이 망가지진 않지만, "이 피처보다는 지난달 평균 구매액 피처를 넣는 게 도메인 관점에서 더 맞지 않나?"처럼 비즈니스적 직관에 따른 조율이 필요할 때 수정을 요청합니다.

APPROVE 판정을 내리는 이유

자동화 검증 수치와 사람의 비즈니스 판단이 모두 부합할 때 비로소 도장을 찍고 컴퓨팅 자원(GPU/CPU)을 투입해 모델을 학습시킵니다.

## 실행 성공 메시지보다 Post-execution Validation 확인


In [22]:
from sklearn.model_selection import train_test_split
import pandas as pd

# 1. 샘플 데이터프레임 생성 (예시)
sample_data = pd.DataFrame(
    {
        "recency_days": [3, 10, 5, 2, 8],
        "total_order_count": [5, 12, 1, 4, 9],
        "target_amount": [150000, 320000, 50000, 120000, 250000],
    }
)

# 2. X(독립변수)와 y(종속변수) 분할
X = sample_data[["recency_days", "total_order_count"]]
y = sample_data["target_amount"]

# 3. Train / Test 데이터셋 분할 -> 여기서 X_train, y_train이 선언됩니다!
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. 데이터 규격 검증 가드레일 실행
try:
    assert (
        X_train.shape[0] == y_train.shape[0]
    ), "❌ [Shape Mismatch] X와 y의 행 개수가 불일치합니다!"
    assert X_train.shape[1] > 0, "❌ [Empty Feature] 피처가 존재하지 않습니다!"
    print("✅ 데이터 규격 검증 통과 (Shape:", X_train.shape, ")")
except AssertionError as e:
    print(f"🚨 전처리 오류 발생: {e}")
    raise

✅ 데이터 규격 검증 통과 (Shape: (4, 2) )
